# Halo, Selamat Pagi!
## In English, it's **Hello, Good Morning!**
### I hope you guys are doing well

My name is **Ezra**, I am currently an IT consultant based in Jakarta, Indonesia. I usually handle projects in *telecommunication* industry
I am so excited to start the journey of learning and practicing LLM engineering, so I want to share my first practice in **week 1** about the **Foundations and the Chat Completions API**

*((Long week ahead until I can learn the Agentic AI in Week 8))*

## Day 1 practice - 4 May 2026
I want to start simple with a straightforward website summarizer. Since I just watched a Persija match (an Indonesia football club), I will summarize **Persija** website for you
Here, I will use **Llama3.2** as the model since it is an open-source model

### Step A: Setting up LLM Libraries and Functions

In [ ]:
#Step 1: Importing the libraries
#Let's learn what are the libraries for, in detail

import os #to allow file & directory manipulation
import requests #to allow HTTP requests in Python
from dotenv import load_dotenv #to read the key-value pairs from .env
from bs4 import BeautifulSoup #to parse HTML & XML docs, useful for webpages navigation
from IPython.display import Markdown, display #to use Markdown function and create rich-text output
from openai import OpenAI #SDK of OpenAi models

#Step 2: Define "openai" function, and need to initialize the Ollama model from our local computer
#NO KEY IS NEEDED, so the .env is not used
#ENSURE YOU INSTALLED OLLAMA FIRST
openai = OpenAI(base_url="http://localhost:11434/v1", api_key='ollama')
#why :11434/v1? let's learn that later

#Step 3: Ensure everything are set up by chatting with Llama
#Define the set of system and user prompts as message_test. SYSTEM FIRST, THEN USER, otherwise will not return anything
#Define the response of the model as response_test
#To print the result, need to choose choices[0].message.content
systemprompt_test = "You act as an administrator that ensure the developer is confident with his code"
userprompt_test = "Hello, Llama! Respond me with 'all good' to ensure that my environment is already set up"
messages_test = [
    {"role":"system", "content":systemprompt_test},
    {"role":"user", "content":userprompt_test}
]
response_test = openai.chat.completions.create(
    model="llama3.2", messages=messages_test
)
print(response_test.choices[0].message.content)

All good


### Step B: Setting Up Website Fetching-Related Functions
*Mostly will just copypaste from Ed's code*

In [ ]:
#Step 1: Define headers, to be used as argument in requests.get function
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

#Step 2: Define Website class
class Website:
    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

#Step 3: Define website content & link fetching functions
def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]
def fetch_website_links(url):
    """
    Return the links on the webiste at the given url
    I realize this is inefficient as we're parsing twice! This is to keep the code in the lab simple.
    Feel free to use a class and optimize it!
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = [link.get("href") for link in soup.find_all("a")]
    return [link for link in links if link]

#Step 4: Test if the scraper is working
fetch_test = fetch_website_contents("https://persija.co.id")
print(fetch_test[:41])
print("Should be working")

Persija Access | Official Fan Engagement 
Should be working


### Step C: AI Model Parameters Definition and the Summarization Result

In [48]:
#Step 1: Set up system_prompt, user_prompt_for, and the messages_for templates
system_prompt = "You are a football expert that summarizes football websites\
by ignoring text that might be navigation-related, and responds in markdown"

#note: the "f" before " appeared automatically since the string is a function
#and we need to separate each lines, while defining the variable increment with +=
#the "website" variable would be the fetch_website_contents() result
def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title()[:20]}" 
    user_prompt += "\n please provide a short summary of this website in markdown.\n"
    user_prompt += "\n and provide the list of players of the club if applicable.\n"
    user_prompt += website
    return user_prompt

#putting it all together in a message content that is readable by OpenAI model
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

#Step 2: Definte the model and summarizer function
#raw summarizer = summarize(url), tidied up summarizer = display_summary(url)
summarizer_model = "llama3.2"
def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model = summarizer_model,
        messages = messages_for(website)
    )
    return response.choices[0].message.content

#add a markdown for tidying up the result
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))
    #Markdown should be capitalized
    #display instead of print: to show rich media content

#Step 3: Define the URL and action! You will see what are the input prompts and 
summarized_url = "https://persija.co.id"
display(messages_for(fetch_website_contents(summarized_url)))
display_summary(summarized_url)
print("---end of summary---")

[{'role': 'system',
  'content': 'You are a football expert that summarizes football websitesby ignoring text that might be navigation-related, and responds in markdown'},
 {'role': 'user',
  'content': "You are looking at a website titled Persija Access | Off\n please provide a short summary of this website in markdown.\n\n and provide the list of players of the club if applicable.\nPersija Access | Official Fan Engagement Platform\n\nOfficial Persija Jakarta Website\nPersija\nJakarta\nHome\nMatches\nNews\nGallery\nVideos\nFirst Team\nStore\n0\nSign In\nPersija\nJakarta\nExperience the legacy of Indonesia's most successful football club. Championing the spirit of Jakarta since 1928.\nDepartment\nHome\nFirst Team\nMatches\nStore\nInsider\nLatest News\nGallery\nVideos\nJoin the Access\nGet priority tickets, exclusive drops, and behind-the-scenes content.\nSubscribe\n© 2024 Persija Jakarta. Developed for Excellence.\nTerms\nPrivacy\nLegal"}]

### Persija Jakarta Overview
#### Football Club Experience
Persija Jakarta is a professional football club based in Jakarta, Indonesia. The website "Persija Access | Official Fan Engagement Platform" seems to be the official fan engagement platform or VIP area of the club.

#### Brief Summary
The Persija Jakarta official fan engagement platform allows fans to access priority tickets, exclusive drops, and behind-the-scenes content. It provides a more personalized experience for loyal supporters of the club.

### List of Notable Players (Not available on this page)
Unfortunately, I couldn't find any information on players listed on this "Persija Access" page. However, according to other sources (not provided here), some notable players from Persija Jakarta's past and current squads include:

*   Muhammad Reza Sajima
*   Andi Efrin Nusembayo
*   Muhammad Arif Sani
*   Irfan Jaya
*   Rizky Mulyadi

Please note that this list is not comprehensive, and players may have changed throughout the years. 

If you want to know more about current players, please visit other reliable sources like:
Persija Jakarta Official Website | Persija.com

---end of summary---


## Day 2 Practice - 4 May 2026

*Self-notes*:
- There are open-source and closed-source
- "Frontier" models refers to closed-source big models. Example: GPT (by OpenAI); o-1,o-2,...,o-5; Claude (by Anthropic); Sonnet 4.5 very powerful; Gemini (by Google) [earlier known as "Bard"]; Grok (by x.ai)
- Open-Source. Example: Llama (by Meta), Mixtral (by Mistral), Qwen (by Alibaba Cloud), Gemma (by Google), Phi (by Microsoft), Deepsek (by DeepSeek AI), GPT-OSS (by OpenAI)
- Smaller models (with small # of parameters, less powerful) usually called "SLM" -> "Small Language Model"
- Three ways to use models: 
- (1) Chat interfaces (like ChatGPT; it's a product that wraps models behind),
- (2) Cloud APIs; LLM API, Frameworks (e.g. LangChain), Managed services (e.g. Amazon Bedrock, Google Vertex, Azure ML)
- (3) Direct inference; with HuggingFace, Transformers library, With Ollama to run locally
